# Stages

## Overview

This page covers repeating part of a computation over many values, such as a list of files.

A workflow often has a part that runs once, such as loading a calibration or building a mask, and a part that runs many times: per file, per detector bank, or per chunk of a stream.
The results of the repeated part are then combined, and the rest of the workflow runs once on the combined value.

Sciline expresses this with [Stage](../generated/classes/sciline.Stage.rst), which cuts a flat pipeline into the part that runs once and the part that runs many times.
Accumulators combine the results of the repeated part, and [enclose](../generated/functions/sciline.enclose.rst) puts stages inside further loops, such as a loop over files around a loop over detector banks.
Nothing is added to the pipeline itself.

## A stage

We start with a pipeline similar to the one in [Getting Started](getting-started.ipynb).
Loading the calibration stands in for an expensive step that does not depend on the file.
The provider appends to a list so we can count how often it runs:

In [ ]:
import math
import operator
from typing import NewType

import sciline

_fake_filesystem = {
    'file102.txt': [1.0, 2.0, float('nan'), 3.0],
    'file103.txt': [1.0, 2.0, 3.0, 4.0],
    'file104.txt': [1.0, 2.0, 3.0, 4.0, 5.0],
    'file105.txt': [1.0, 2.0, 3.0],
    'calib_2025.txt': [0.5],
    'calib_2026.txt': [0.25],
}

# 1. Define domain types

Filename = NewType('Filename', str)
RawData = NewType('RawData', list)
CalibrationFilename = NewType('CalibrationFilename', str)
Calibration = NewType('Calibration', float)
Calibrated = NewType('Calibrated', list)
Total = NewType('Total', float)
ScaleFactor = NewType('ScaleFactor', float)
Result = NewType('Result', float)

# 2. Define providers

calibration_loads = []


def load(filename: Filename) -> RawData:
    """Load the data from a file."""
    return RawData(_fake_filesystem[filename])


def load_calibration(filename: CalibrationFilename) -> Calibration:
    """Load the calibration factor."""
    calibration_loads.append(filename)
    return Calibration(_fake_filesystem[filename][0])


def calibrate(raw: RawData, calibration: Calibration) -> Calibrated:
    """Drop NaNs and apply the calibration."""
    return Calibrated([x * calibration for x in raw if not math.isnan(x)])


def total(data: Calibrated) -> Total:
    """Sum the calibrated data."""
    return Total(sum(data))


def scale(total: Total, factor: ScaleFactor) -> Result:
    """Scale the total."""
    return Result(total * factor)


# 3. Create pipeline

providers = [load, load_calibration, calibrate, total, scale]
params = {CalibrationFilename: 'calib_2025.txt', ScaleFactor: 2.0}
pipeline = sciline.Pipeline(providers, params=params)
pipeline.visualize(Result, graph_attr={'rankdir': 'LR'})

`Filename` has no value.
A [Stage](../generated/classes/sciline.Stage.rst) cuts the pipeline at a set of *inputs*, keys whose values are supplied on each call, and computes the values of a set of *outputs* from them.
Everything the outputs need that does not depend on the inputs is computed once, on first use.
The stage holds the values of this part that the per-call part reads, its *frontier*:

In [ ]:
stage = sciline.Stage(pipeline, outputs=(Result,), inputs=(Filename,))
{
    f: stage.compute({Filename: f})[Result]
    for f in ['file102.txt', 'file103.txt', 'file104.txt']
}

`compute` was called three times, but the calibration was loaded once:

In [ ]:
calibration_loads

The stage tells us where it cut the pipeline.
`visualize` draws the part of the pipeline the stage uses, with the inputs, the part computed once, the held values, and the per-call part marked:

In [ ]:
stage.visualize(graph_attr={'rankdir': 'TD'})

`frontier` and `dynamic` are tuples of keys and `keys` is a frozenset.
`frontier` lists the held keys, and `static()` returns their values.
`dynamic` lists the keys whose values depend on the inputs and are computed on each call.
`keys` contains every key the stage uses, computed once or per call:

In [ ]:
stage.frontier

In [ ]:
stage.dynamic

In [ ]:
stage.keys

An input does not have to be a parameter.
If it is an intermediate result, its provider and everything upstream of it are cut off, and the stage computes only what lies downstream.
Here `Total` is supplied, so nothing is loaded and only `ScaleFactor` is held; the cut-off keys are not in `keys` either:

In [ ]:
from_total = sciline.Stage(pipeline, outputs=(Result,), inputs=(Total,))
from_total.frontier, from_total.dynamic, from_total.keys

In [ ]:
from_total.compute({Total: 10.0})

A stage is a snapshot of the pipeline at the time it is built.
Set all parameters, except the inputs, before building it; changing the pipeline afterwards does not affect the stage.

## Warming several stages

Two stages built from the same pipeline may share work that each computes once.
Each stage computes this part on first use, so two stages used independently both load the calibration.
[warm](../generated/functions/sciline.warm.rst) computes this part of several stages in one run, so shared work is done once:

In [ ]:
calibration_loads.clear()
calibrated = sciline.Stage(pipeline, outputs=(Calibrated,), inputs=(Filename,))
totals = sciline.Stage(pipeline, outputs=(Total,), inputs=(Filename,))
sciline.warm(calibrated, totals)
calibration_loads

Each stage holds only the values at its own frontier, available from `static()`:

In [ ]:
calibrated.static(), totals.static()

## Combining over files

To combine over files, we need two stages: one from `Filename` to the key at which we combine, the *accumulation key*, and one from that key to the result.
Between them, an accumulator combines the values computed per file.
An accumulator is any object with a `push` method and a `value` property, see [Accumulator](../generated/classes/sciline.Accumulator.rst).

Here we combine `Calibrated` by concatenation.
[Buffered](../generated/classes/sciline.Buffered.rst) makes accumulators that keep every pushed value and apply a function to all of them when the combined value is read:

In [ ]:
def concat(*lists: list) -> list:
    return [x for values in lists for x in values]


def combine_files(contribute, finalize, accumulators, filenames):
    acc = {key: make() for key, make in accumulators.items()}
    for filename in filenames:
        values = contribute.compute({Filename: filename})
        for key in contribute.dynamic_outputs:
            acc[key].push(values[key])
    return finalize.compute({key: a.value for key, a in acc.items()})


files = ['file102.txt', 'file103.txt', 'file104.txt', 'file105.txt']
contribute = sciline.Stage(pipeline, outputs=(Calibrated,), inputs=(Filename,))
finalize = sciline.Stage(pipeline, outputs=(Result,), inputs=(Calibrated,))
combine_files(contribute, finalize, {Calibrated: sciline.Buffered(concat)}, files)

`Calibrated` was computed per file, the four lists were concatenated, and `Total` and `Result` were computed once from the concatenated list.
`combine_files` is the *driver*: the code that calls the stages, here in a loop over files, and pushes their values into accumulators.
It pushes `dynamic_outputs`, the outputs that depend on the inputs.
The section "Push only what depends on the files" below explains why.
`Buffered(concat)` is a factory: calling it gives a new, empty accumulator.
`combine_files` calls the factories each time it runs, so a second call does not add to the values of the first.

`Buffered` keeps every pushed value.
For a sum of large arrays over many files, a running total is much cheaper.
[Reduced](../generated/classes/sciline.Reduced.rst) makes accumulators that apply a binary function to the result so far and each pushed value, and keep only the result.
`Reduced(operator.add)` is a factory too.
The function must be associative: a combined value can be pushed again, so values may be combined in groups, see "The steps separately".
It must not modify its arguments.
Here we combine at `Total` instead, and sum as we go:

In [ ]:
contribute = sciline.Stage(pipeline, outputs=(Total,), inputs=(Filename,))
finalize = sciline.Stage(pipeline, outputs=(Result,), inputs=(Total,))
combine_files(contribute, finalize, {Total: sciline.Reduced(operator.add)}, files)

The result is the same, but only one number per file is pushed, and the accumulator keeps only the running total.

### Push only what depends on the files

`combine_files` pushes `dynamic_outputs`, not all outputs of the stage.
An output that does not depend on the inputs, such as `Calibration`, is computed once and returned by every call.
Pushed into a sum, it would be counted once per file:

In [ ]:
with_calibration = sciline.Stage(
    pipeline, outputs=(Total, Calibration), inputs=(Filename,)
)
with_calibration.outputs, with_calibration.dynamic_outputs

The final stage takes only the dynamic outputs as inputs.
It holds `Calibration`, computed once:

In [ ]:
finalize_with_calibration = sciline.Stage(
    pipeline,
    outputs=(Result, Calibration),
    inputs=with_calibration.dynamic_outputs,
)
combine_files(
    with_calibration,
    finalize_with_calibration,
    {Total: sciline.Reduced(operator.add)},
    files,
)

## The steps separately

The loop in `combine_files` has three steps: compute a contribution per file, push its values into accumulators, and compute the result from the combined values.
They can also be called separately.
A contribution is a plain dictionary, so the steps can run in separate processes, with the contributions serialized between them:

In [ ]:
def combine(contributions):
    acc = sciline.Reduced(operator.add)()
    for contribution in contributions:
        acc.push(contribution[Total])
    return {Total: acc.value}


contributions = {f: contribute.compute({Filename: f}) for f in files}
finalize.compute(combine(contributions.values()))

Combined values can be pushed again, so combining can also proceed in groups.
The driver can keep the contributions in a dict keyed by file, to add or remove files without recomputing the others.
Here we drop one file:

In [ ]:
del contributions['file105.txt']
finalize.compute(combine(contributions.values()))

## Files in parallel

`contribute.compute` is a plain function of the values of the inputs, so any executor can run it.
Sciline does not run anything in parallel itself; the driver decides how to iterate over the files.
We warm the stages first, so that the threads do not each compute the part computed once:

In [ ]:
from concurrent.futures import ThreadPoolExecutor

sciline.warm(contribute, finalize)
with ThreadPoolExecutor() as executor:
    contributions = executor.map(lambda f: contribute.compute({Filename: f}), files)
    combined = combine(contributions)
finalize.compute(combined)

## Results per file without combining

To compute the value of a key for each file without combining, call a stage in a loop.
The `calibrated` stage from above holds the calibration and computes `Calibrated` per file:

In [ ]:
{f: calibrated.compute({Filename: f})[Calibrated] for f in files}

## Nested loops

Suppose each file contains two detector banks, and reading a file is expensive.
We replace `load` by two providers: one reads the file, one selects a bank from it:

In [ ]:
Bank = NewType('Bank', int)
FileContent = NewType('FileContent', list)

file_reads = []


def read_file(filename: Filename) -> FileContent:
    """Read all banks of a file."""
    file_reads.append(filename)
    return FileContent(_fake_filesystem[filename])


def select_bank(content: FileContent, bank: Bank) -> RawData:
    """Even values are in bank 0, odd values in bank 1."""
    return RawData(content[bank::2])


banked = pipeline.copy()
banked.insert(read_file)
banked.insert(select_bank)

We want `Total` summed over all banks of all files, reading each file once.
Written as plain loops that call the providers by hand, the computation is below.
The comments name the stage that computes each line; we build these stages in this section.

```python
calibration = load_calibration(calibration_filename)  # held by bank_stage
summed = 0.0  # accumulator
for filename in filenames:
    content = read_file(filename)  # file_stage
    for b in banks:
        raw = select_bank(content, b)  # bank_stage
        summed += total(calibrate(raw, calibration))  # bank_stage, pushed
result = scale(summed, scale_factor)  # finalize_banked
```

A single stage from `(Filename, Bank)` to `Total` would do the work of both loops in each call, and read each file once per bank.
Instead, we build a stage per loop, from the inside out.
The inner loop computes `Total` for one bank, so we start with a stage from `Bank` to `Total`:

In [ ]:
bank_stage = sciline.Stage(banked, outputs=(Total,), inputs=(Bank,))
bank_stage.frontier

The stage holds `FileContent` and `Calibration`.
Called once per bank, it would see the banks of only one file, the one set on the pipeline; here no file is set.
To loop over files as well, we put the stage inside a loop over `Filename` with [enclose](../generated/functions/sciline.enclose.rst):

In [ ]:
file_stage, bank_stage = sciline.enclose(banked, [bank_stage], inputs=(Filename,))
file_stage.outputs, bank_stage.inputs

`enclose` returns the outer stage, which the driver calls once per file, followed by the rebuilt bank stage.
Of the values that the bank stage held, the outer stage computes those that depend on `Filename`, once per file.
The rebuilt bank stage takes them as inputs.
`Calibration` depends on neither loop, so the bank stage still holds it.

The rules of `enclose`:

- Pass every stage that runs inside the new loop.
  A stage that runs after the loop, such as `finalize_banked` below, is built with `Stage` and not passed.
- It returns the outer stage, which the driver calls once per iteration of the new loop, followed by the passed stages, rebuilt, in the order given.
- The outer stage outputs the values that the passed stages held and that depend on the new inputs.
  The rebuilt stages take them as inputs instead of holding them.
- A loop cannot be extended later.
  To add a stage to the file loop, enclose all its stages again, starting from the stages as built with `Stage`.
- Wrapping a further loop around existing loops is fine: pass all their stages.

[visualize_stages](../generated/functions/sciline.visualize_stages.rst) draws the stages together, with a color for what each stage computes per call.
The legend numbers the stages in the order of the arguments, starting from "Stage 0":

In [ ]:
finalize_banked = sciline.Stage(banked, outputs=(Result,), inputs=(Total,))
sciline.visualize_stages(
    file_stage, bank_stage, finalize_banked, graph_attr={'rankdir': 'TD'}
)

The driver has the shape of the loops above.
Each stage gets everything that the stages of the enclosing loops returned, plus its own loop value.
Here the bank stage gets what the file stage returned, plus the bank.
A stage ignores values for keys it does not use, so the driver does not have to select them.

The driver makes a new accumulator per iteration of the loop that the accumulation belongs to.
Here we sum over all files and banks, so there is one accumulator per call of the driver; `compute_means` below makes one per file.

Before running the driver, we pass all its stages to `warm`.
Shared work is then computed once, and `warm` checks that the stages fit together, see "What enclose refuses":

In [ ]:
def compute_banked(filenames, banks):
    summed = sciline.Reduced(operator.add)()
    for filename in filenames:
        per_file = file_stage.compute({Filename: filename})
        for b in banks:
            values = bank_stage.compute({**per_file, Bank: b})
            summed.push(values[Total])
    return finalize_banked.compute({Total: summed.value})


file_reads.clear()
sciline.warm(file_stage, bank_stage, finalize_banked)
compute_banked(files, banks=[0, 1])

Each file was read once:

In [ ]:
file_reads

### Work after combining the banks

Suppose we want a mean per file: the total of the banks of a file, divided by the number of valid values in the file.
As plain loops:

```python
means = {}
for filename in filenames:
    content = read_file(filename)  # file_stage
    n = num_values(content)  # file_stage
    summed = 0.0  # accumulator, one per file
    for b in banks:
        raw = select_bank(content, b)  # bank_stage
        summed += total(calibrate(raw, calibration))  # bank_stage, pushed
    means[filename] = mean(summed, n)  # mean_stage
```

The division runs once per file, after the bank loop.
Its stage goes from `Total`, the accumulation key of the bank loop, to `Mean`.
It holds `NumValues`, which depends on the file:

In [ ]:
NumValues = NewType('NumValues', int)
Mean = NewType('Mean', float)


def num_values(content: FileContent) -> NumValues:
    """Count the valid values of all banks."""
    return NumValues(sum(1 for x in content if not math.isnan(x)))


def mean(total: Total, n: NumValues) -> Mean:
    return Mean(total / n)


banked.insert(num_values)
banked.insert(mean)
mean_stage = sciline.Stage(banked, outputs=(Mean,), inputs=(Total,))
mean_stage.frontier

The division is inside the file loop, like the bank loop.
To add the mean stage to the file loop, we rebuild the loop, following the rules above.
The bank stage that the first `enclose` returned takes `FileContent` as input instead of holding it.
Enclosed again, it would not make the file stage output `FileContent`, and the driver call of the bank stage would fail with a missing input.
So we enclose both stages in one call, starting again from the stage from `Bank` to `Total` as built with `Stage`.
The file stage then computes what either of them reads:

In [ ]:
bank_stage = sciline.Stage(banked, outputs=(Total,), inputs=(Bank,))
file_stage, bank_stage, mean_stage = sciline.enclose(
    banked, [bank_stage, mean_stage], inputs=(Filename,)
)
file_stage.outputs, mean_stage.inputs

In [ ]:
sciline.visualize_stages(
    file_stage, bank_stage, mean_stage, graph_attr={'rankdir': 'TD'}
)

The driver follows the loops above, with a new accumulator for each file:

In [ ]:
def compute_means(filenames, banks):
    means = {}
    for filename in filenames:
        per_file = file_stage.compute({Filename: filename})
        summed = sciline.Reduced(operator.add)()
        for b in banks:
            values = bank_stage.compute({**per_file, Bank: b})
            summed.push(values[Total])
        values = mean_stage.compute({**per_file, Total: summed.value})
        means[filename] = values[Mean]
    return means


sciline.warm(file_stage, bank_stage, mean_stage)
compute_means(files, banks=[0, 1])

`per_file` contains `FileContent` and `NumValues`.
The bank stage uses only the former, and the mean stage only the latter.

### Three loops

Suppose we want to compare the means for two calibrations.
This puts a loop over calibration files around the loop over files:

```python
for calibration_filename in calibration_filenames:
    calibration = load_calibration(calibration_filename)  # calibration_stage
    for filename in filenames:
        ...  # as above
```

`Calibration` is held by the bank stage, and it depends on `CalibrationFilename`.
Enclosing the stages in a loop over `CalibrationFilename` moves it to the outer stage of the calibration loop.
We pass every stage that runs inside the calibration loop, not only the file stage.
Here the bank stage is the one that reads `Calibration`.
The value of `CalibrationFilename` set on the pipeline is not used once it is an input.
The driver supplies it:

In [ ]:
calibration_stage, file_stage, bank_stage, mean_stage = sciline.enclose(
    banked, [file_stage, bank_stage, mean_stage], inputs=(CalibrationFilename,)
)
calibration_stage.outputs, bank_stage.inputs

In [ ]:
sciline.visualize_stages(
    calibration_stage,
    file_stage,
    bank_stage,
    mean_stage,
    graph_attr={'rankdir': 'TD'},
)

The driver has one more loop.
As before, each stage gets everything that the stages of the enclosing loops returned, plus its own loop value:

In [ ]:
def compare_calibrations(calibration_filenames, filenames, banks):
    means = {}
    for calibration_filename in calibration_filenames:
        per_calibration = calibration_stage.compute(
            {CalibrationFilename: calibration_filename}
        )
        for filename in filenames:
            per_file = file_stage.compute({**per_calibration, Filename: filename})
            summed = sciline.Reduced(operator.add)()
            for b in banks:
                values = bank_stage.compute({**per_calibration, **per_file, Bank: b})
                summed.push(values[Total])
            values = mean_stage.compute(
                {**per_calibration, **per_file, Total: summed.value}
            )
            means[calibration_filename, filename] = values[Mean]
    return means


calibration_loads.clear()
file_reads.clear()
sciline.warm(calibration_stage, file_stage, bank_stage, mean_stage)
compare_calibrations(['calib_2025.txt', 'calib_2026.txt'], files, banks=[0, 1])

Each calibration was loaded once:

In [ ]:
calibration_loads

Each file was read once per calibration:

In [ ]:
file_reads

With the calibration loop inside the file loop, each file is read once, but each calibration is loaded once per file.
Choose the order of the loops by which step is more expensive.

### What enclose refuses

Suppose the bank stage also outputs `NumValues`.
Before enclosing, `NumValues` is held, so it is not in `dynamic_outputs` and a driver does not push it.
After enclosing, it would be an input of the bank stage, returned as an output.
The driver would push it once per bank and count it twice.
`enclose` refuses such a stage:

In [ ]:
counting = sciline.Stage(banked, outputs=(Total, NumValues), inputs=(Bank,))
try:
    sciline.enclose(banked, [counting], inputs=(Filename,))
except ValueError as error:
    print(error)

The fix that the message names: the bank stage outputs only `Total`, and the outer stage outputs `NumValues`, which the driver then pushes once per file:

In [ ]:
bank_total = sciline.Stage(banked, outputs=(Total,), inputs=(Bank,))
file_counting, bank_total = sciline.enclose(
    banked, [bank_total], inputs=(Filename,), outputs=(NumValues,)
)
file_counting.outputs

A stage that we forget to pass to `enclose` keeps holding the values that depend on the inputs of the loop.
If a file is set on the pipeline, it holds the values for that file, and the driver gives wrong results without an error.
Here no file is set, so the stage fails when computing.
[warm](../generated/functions/sciline.warm.rst), given all stages of the driver, rejects a stage that holds a value depending on a parameter that another stage takes as input:

In [ ]:
bank_only = sciline.Stage(banked, outputs=(Total,), inputs=(Bank,))
forgotten = sciline.Stage(banked, outputs=(Mean,), inputs=(Total,))
outer, bank_only = sciline.enclose(banked, [bank_only], inputs=(Filename,))
try:
    sciline.warm(outer, bank_only, forgotten)
except ValueError as error:
    print(error)